In [ ]:
!pip install transformers datasets accelerate torchvision scikit-learn nltk evaluate sentencepiece sacrebleu
!pip install git+https://github.com/salesforce/BLIP.git


In [ ]:
import os
import random
import torch
import nltk
import evaluate
from PIL import Image
from torchvision import transforms
from datasets import load_dataset, Dataset, DatasetDict
from transformers import Blip2Processor, Blip2ForConditionalGeneration, TrainingArguments, Trainer, default_data_collator
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
nltk.download('punkt')


In [ ]:
flickr_dataset = load_dataset("csv", data_files={"train": "https://huggingface.co/datasets/nlpconnect/flickr8k/resolve/main/flickr8k.csv"})
images_path = "https://huggingface.co/datasets/nlpconnect/flickr8k/resolve/main/images/"

processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

def preprocess(example):
    image_url = images_path + example["image"]
    image = Image.open(requests.get(image_url, stream=True).raw).convert("RGB")
    image = transform(image)
    example["pixel_values"] = image
    example["text"] = example["caption"]
    return example

small_dataset = flickr_dataset["train"].shuffle(seed=42).select(range(5000))
processed_dataset = small_dataset.map(preprocess, remove_columns=small_dataset.column_names)
train_data, val_data = train_test_split(processed_dataset, test_size=0.1)
dataset = DatasetDict({"train": Dataset.from_dict(train_data), "validation": Dataset.from_dict(val_data)})


In [ ]:
model = Blip2ForConditionalGeneration.from_pretrained("Salesforce/blip2-opt-2.7b", device_map="auto", torch_dtype=torch.float16)


In [ ]:
training_args = TrainingArguments(
    output_dir="./blip2-finetuned",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=5e-5,
    logging_dir="./logs",
    logging_steps=50,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    fp16=True,
    report_to="none"
)

def collate_fn(batch):
    pixel_values = torch.stack([x["pixel_values"] for x in batch])
    captions = [x["text"] for x in batch]
    inputs = processor(images=pixel_values, text=captions, return_tensors="pt", padding=True)
    inputs["labels"] = inputs["input_ids"]
    return inputs

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    data_collator=collate_fn,
)

trainer.train()


In [ ]:
def generate_caption(image, strategy="greedy", **kwargs):
    inputs = processor(images=image, return_tensors="pt").to("cuda", torch.float16)
    if strategy == "beam":
        output = model.generate(**inputs, num_beams=5, max_new_tokens=50)
    elif strategy == "top_k":
        output = model.generate(**inputs, do_sample=True, top_k=50, max_new_tokens=50)
    elif strategy == "top_p":
        output = model.generate(**inputs, do_sample=True, top_p=0.9, max_new_tokens=50)
    elif strategy == "temperature":
        output = model.generate(**inputs, do_sample=True, temperature=1.2, max_new_tokens=50)
    else:
        output = model.generate(**inputs, max_new_tokens=50)
    return processor.decode(output[0], skip_special_tokens=True)


In [ ]:
bleu = evaluate.load("bleu")
meteor = evaluate.load("meteor")
rouge = evaluate.load("rouge")
sacrebleu = evaluate.load("sacrebleu")

def compute_metrics(preds, labels):
    results = {}
    results["bleu"] = bleu.compute(predictions=preds, references=labels)["bleu"]
    results["meteor"] = meteor.compute(predictions=preds, references=labels)["meteor"]
    results["rougeL"] = rouge.compute(predictions=preds, references=labels)["rougeL"]
    results["sacrebleu"] = sacrebleu.compute(predictions=preds, references=[[ref] for ref in labels])["score"]
    return results


In [ ]:
val_samples = dataset["validation"].select(range(100))
preds = []
labels = []

for sample in val_samples:
    image = sample["pixel_values"]
    label = sample["text"]
    pred = generate_caption(image=image, strategy="beam")
    preds.append(pred)
    labels.append(label)

metrics = compute_metrics(preds, labels)
print(metrics)


In [ ]:
import matplotlib.pyplot as plt

samples = dataset["validation"].select(range(20))

for sample in samples:
    image = sample["pixel_values"]
    label = sample["text"]
    pred = generate_caption(image=image, strategy="top_p")
    plt.imshow(transforms.ToPILImage()(image))
    plt.title(f"Predicted: {pred}\nActual: {label}")
    plt.axis("off")
    plt.show()


In [ ]:
model.save_pretrained("./blip2-finetuned")
processor.save_pretrained("./blip2-finetuned")
